# Exercício: Gerenciamento de Memória na Prática
## Parte A - Implementação Manual Ingênua
Este notebook contém a resolução da **Parte A** adaptada para os dados médicos gerados (colunas: `timestamp`, `estacao_id`, `sensor_id`, `bpm`, `spo2_pct`, `temperatura_c`, `pressao_sistolica`, `pressao_diastolica`).

### A.2 Leitura do Arquivo
Carregando o arquivo inteiro de uma vez em uma lista de dicionários (abordagem ingênua).

In [ ]:
import csv

def carregar_leituras(caminho_arquivo):
    leituras = []
    # A inclusão do newline='' previne erros de leitura no Windows que unem linhas acidentalmente
    with open(caminho_arquivo, mode='r', encoding='utf-8', newline='') as f:
        leitor = csv.DictReader(f)
        for linha in leitor:
            # Conversão de tipos
            linha['bpm'] = int(linha['bpm'])
            linha['spo2_pct'] = int(linha['spo2_pct'])
            linha['temperatura_c'] = float(linha['temperatura_c'])
            linha['pressao_sistolica'] = int(linha['pressao_sistolica'])
            linha['pressao_diastolica'] = int(linha['pressao_diastolica'])
            leituras.append(linha)
    return leituras

### A.3 Operações a Implementar

In [ ]:
def operacao_contagens(leituras):
    estacoes, sensores = {}, {}
    for lin in leituras:
        estacoes[lin['estacao_id']] = estacoes.get(lin['estacao_id'], 0) + 1
        sensores[lin['sensor_id']] = sensores.get(lin['sensor_id'], 0) + 1
        
    return len(leituras), estacoes, sensores


def operacao_agregacoes(leituras):
    st_est, st_sen = {}, {}
    
    for lin in leituras:
        est, sen = lin['estacao_id'], lin['sensor_id']
        temp, bpm, spo2 = lin['temperatura_c'], lin['bpm'], lin['spo2_pct']
        
        # Inicializa as chaves se for a primeira vez que vemos a estação/sensor
        if est not in st_est:
            st_est[est] = {'soma_temp': 0, 'count': 0, 'min_temp': temp, 'max_temp': temp, 'spo2_crit': 0}
        if sen not in st_sen:
            st_sen[sen] = {'soma_bpm': 0, 'count': 0, 'min_bpm': bpm, 'max_bpm': bpm}
            
        e, s = st_est[est], st_sen[sen]
        
        # Atualiza Estação
        e['soma_temp'] += temp
        e['count'] += 1
        e['min_temp'] = min(e['min_temp'], temp)
        e['max_temp'] = max(e['max_temp'], temp)
        e['spo2_crit'] += 1 if spo2 < 92 else 0
        
        # Atualiza Sensor
        s['soma_bpm'] += bpm
        s['count'] += 1
        s['min_bpm'] = min(s['min_bpm'], bpm)
        s['max_bpm'] = max(s['max_bpm'], bpm)
        
    # Calcula as médias
    for stats in st_est.values():
        stats['media_temp'] = stats['soma_temp'] / stats['count']
        stats['pct_spo2_crit'] = (stats['spo2_crit'] / stats['count']) * 100
        
    for stats in st_sen.values():
        stats['media_bpm'] = stats['soma_bpm'] / stats['count']
        stats['var_bpm'] = stats['max_bpm'] - stats['min_bpm']
        
    return st_est, st_sen


def operacao_novas_colunas_e_salvar(leituras, caminho_saida):
    for lin in leituras:
        # PAM
        lin['pressao_media'] = round((lin['pressao_sistolica'] + 2 * lin['pressao_diastolica']) / 3, 1)
        
        # Status
        spo2 = lin['spo2_pct']
        lin['status_paciente'] = 'critica' if spo2 < 92 else 'alerta' if spo2 < 95 else 'normal'

    if leituras:
        with open(caminho_saida, 'w', newline='', encoding='utf-8') as f:
            escritor = csv.DictWriter(f, fieldnames=leituras[0].keys())
            escritor.writeheader()
            escritor.writerows(leituras)

### A.4 Medição de Memória e Tempo

In [ ]:
import tracemalloc
import time
import gc

def medir_pipeline(arquivo):
    gc.collect()
    res = {}
    t0 = time.time()
    
    # 1. Carga
    tracemalloc.start()
    leituras = carregar_leituras(arquivo)
    res['carga_mb'] = tracemalloc.get_traced_memory()[1] / 1024**2
    tracemalloc.stop()
    
    # 2. Agregações
    tracemalloc.start()
    operacao_contagens(leituras)
    operacao_agregacoes(leituras)
    res['agreg_mb'] = tracemalloc.get_traced_memory()[1] / 1024**2
    tracemalloc.stop()
    
    # 3. Escrita
    tracemalloc.start()
    operacao_novas_colunas_e_salvar(leituras, arquivo.replace('.csv', '_proc.csv'))
    res['escr_mb'] = tracemalloc.get_traced_memory()[1] / 1024**2
    tracemalloc.stop()
    
    res['tempo'] = time.time() - t0
    
    # Limpeza para o próximo ciclo
    del leituras
    gc.collect()
    
    return res

arquivos = ['leituras_sensores_100k.csv', 'leituras_sensores_1m.csv', 'leituras_sensores_5m.csv']

print(f"{'Arquivo':<28} | {'Carga(MB)':<10} | {'Agreg(MB)':<10} | {'Escr(MB)':<10} | {'Tempo(s)':<8}")
print("-" * 75)

for arq in arquivos:
    try:
        r = medir_pipeline(arq)
        print(f"{arq:<28} | {r['carga_mb']:>10.2f} | {r['agreg_mb']:>10.2f} | {r['escr_mb']:>10.2f} | {r['tempo']:>8.2f}")
    except FileNotFoundError:
        print(f"{arq:<28} | Arquivo não encontrado")
    except MemoryError:
        print(f"{arq:<28} | ❌ Erro: Memória RAM estourou!")
    except Exception as e:
        print(f"{arq:<28} | Erro: {e}")

### A.5 Diagnóstico

**Pergunta: O que acontece com o pico de memória à medida que o arquivo cresce? A relação é aproximadamente linear? Por quê?**
Sim, a relação é estritamente linear ($O(N)$). O pico de memória na fase de carga cresce proporcionalmente ao número de linhas do arquivo. Isso ocorre porque o código instancia e adiciona à memória RAM um novo dicionário Python inteiro para cada linha lida no arquivo. 

**Três pontos de desperdício/redundância na versão ingênua:**
1. **Carregamento integral em Lista de Dicionários:** Manter milhões de dicionários na memória simultaneamente consome muito mais memória do que o arquivo em disco. Dicionários em Python possuem um *overhead* interno considerável.
2. **Repetição excessiva de Strings em Memória:** Os IDs lógicos (`estacao_id`, `sensor_id`) e os nomes das chaves do dicionário se repetem milhões de vezes sem *interning* explícito.
3. **Iterações e Estruturas Redundantes:** Nas operações, iteramos a lista original repetidas vezes, e ao escrever o novo arquivo, mantemos tudo na memória em vez de gravar em lote (streaming).